In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import os.path as op
import yaml
import pandas as pd

In [ ]:


def read_all_results_yaml(root_dir, select = ('netmap_config.yaml', 'netmap_config_1.yaml')):
    """
    Reads all YAML files in a directory tree starting from root_dir, processes
    the nested structure (including multiple top-level keys), and concatenates
    the results into a single DataFrame.

    Args:
        root_dir (str): The path to the root directory.
        select (tuple or str): File extension or full filename to select.
                               Defaults to common YAML configuration names.

    Returns:
        pandas.DataFrame: A DataFrame containing the processed data from all YAML files.
                          Returns an empty DataFrame if no files are found or an error occurs.
    """
    all_results_dfs = []

    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return pd.DataFrame() # Return an empty DataFrame

    if isinstance(select, str):
        select = (select,)

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            # Check if the filename ends with any of the selected strings/extensions
            if any(filename.endswith(s) for s in select):
                filepath = op.join(dirpath, filename)
                print(f"Processing: {filepath}")

                try:
                    with open(filepath, 'r') as f:
                        yaml_content = yaml.safe_load(f)

                    if not yaml_content or not isinstance(yaml_content, dict):
                        continue # Skip empty or invalid YAML file

                    # The directory one level up is 'CONFIG'
                    config_dir = op.basename(op.dirname(dirpath))
                    
                    # Iterate through all top-level keys (e.g., netmap_config_1, netmap_config_2, etc.)
                    for config_key, nested_data in yaml_content.items():
                        
                        if not isinstance(nested_data, dict):
                            print(f"Skipping key '{config_key}' in {filename}: not a dictionary.")
                            continue

                        records = []
                        
                        # Extract common metadata for this configuration key
                        clustering_score = nested_data.get('clustering_score')
                        total_number_edges = nested_data.get('total_number_edges')

                        # Iterate through the network-specific keys (like net_53_1113)
                        for net_key, net_data in nested_data.items():
                            
                            # Skip known non-network metadata fields
                            if net_key in ['clustering_score', 'total_number_edges']:
                                continue
                            
                            if not isinstance(net_data, dict):
                                print(f"Skipping network key '{net_key}' under '{config_key}': not a dictionary.")
                                continue

                            # Create a record for the DataFrame
                            record = {
                                'net_name': net_key,           # e.g., net_53_1113
                                'config_key': config_key,      # e.g., netmap_config_1
                                'filename': filename,
                                # Add the common metadata
                                'clustering_score': clustering_score,
                                'total_number_edges': total_number_edges,
                            }
                            # Add the network-specific data (edge_overlap, net_size, etc.)
                            record.update(net_data)
                            
                            records.append(record)

                        # Create a DataFrame from the extracted records for this config_key
                        if records:
                            net_dir = op.basename(dirpath)  
                            # The directory one level up is 'CONFIG'
                            config_dir = op.basename(op.dirname(dirpath))
                        


                            overlaps_df = pd.DataFrame(records)
                            # Add the directory name (similar to your original 'net' column)
                            overlaps_df['dataset'] = op.basename(op.dirname(filepath))
                            overlaps_df['config_dir'] = config_dir
                            all_results_dfs.append(overlaps_df)


                except Exception as e:
                    print(f"Error reading or processing {filepath}: {e}")
                    continue

    if all_results_dfs:
        # Concatenate all DataFrames into one
        all_results = pd.concat(all_results_dfs, ignore_index=True)
    else:
        all_results = pd.DataFrame()

    all_results['n_clusters'] = all_results['config_dir'].apply(recode_number_of_datasets)


    return all_results

def recode_number_of_datasets(config_item):
    if config_item == 'config_easy':
        return 2
    if config_item == 'config_noise':
        return 2
    if config_item == 'config_three':
        return 3
    if config_item == 'config_three_noise':
        return 3
    if config_item == 'config_five':
        return 5
    if config_item == 'config_ten':
        return 10


In [16]:
df_results_leiden = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/', select=('clustering_score_leiden.json'))
df_results_leiden.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_scores_leiden.tsv', sep = '\t', index=False)

df_results = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/', select=('clustering_score.json'))
df_results.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_score.tsv', sep = '\t', index = False)

df_results_tf = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/', select=('clustering_score_tf.json'))
df_results_tf.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_score_tf.tsv', sep = '\t', index = False)

df_results_leiden = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_scgenerai/', select=('clustering_score_leiden.json'))
df_results_leiden.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_scores_scgenerai.tsv', sep = '\t', index=False)



Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/net_84_10865_net_88_10937_net_90_11013/net_84_10865_net_88_10937_net_90_11013.yaml
Skipping key 'network' in net_84_10865_net_88_10937_net_90_11013.yaml: not a dictionary.
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/net_84_10865_net_88_10937_net_90_11013/64_1_0.1_NegativeBinomialAutoencoder_training_time.json
Skipping key 'training_time' in 64_1_0.1_NegativeBinomialAutoencoder_training_time.json: not a dictionary.
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/net_84_10865_net_88_10937_net_90_11013/GradientShap_64_1_0.1_NegativeBinomialAutoencoder_zeros_False_grn.h5ad
Error reading or processing /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/net_84_10865_net_88_10937_net_90_11013/GradientShap_64_1_0.1_NegativeBinomialAutoencoder_zeros_False_grn.h5ad: 'utf-8' codec can't decod

In [ ]:


def read_all_results_yaml(root_dir, select = ('netmap_config.yaml', 'netmap_config_1.yaml')):
    """
    Reads all YAML files in a directory tree starting from root_dir, processes
    the nested structure (including multiple top-level keys), and concatenates
    the results into a single DataFrame.

    Args:
        root_dir (str): The path to the root directory.
        select (tuple or str): File extension or full filename to select.
                               Defaults to common YAML configuration names.

    Returns:
        pandas.DataFrame: A DataFrame containing the processed data from all YAML files.
                          Returns an empty DataFrame if no files are found or an error occurs.
    """
    all_results_dfs = []

    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return pd.DataFrame() # Return an empty DataFrame

    if isinstance(select, str):
        select = (select,)

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:

            filepath = op.join(dirpath, filename)
            print(f"Processing: {filepath}")

            try:
                with open(filepath, 'r') as f:
                    yaml_content = yaml.safe_load(f)

                if not yaml_content or not isinstance(yaml_content, dict):
                    continue # Skip empty or invalid YAML file

                # The directory one level up is 'CONFIG'
                config_dir = op.basename(op.dirname(dirpath))
                
                # Iterate through all top-level keys (e.g., netmap_config_1, netmap_config_2, etc.)
                for config_key, nested_data in yaml_content.items():
                    
                    if not isinstance(nested_data, dict):
                        print(f"Skipping key '{config_key}' in {filename}: not a dictionary.")
                        continue

                    records = []
                    
                    # Extract common metadata for this configuration key
                    clustering_score = nested_data.get('clustering_score')
                    total_number_edges = nested_data.get('total_number_edges')

                    # Iterate through the network-specific keys (like net_53_1113)
                    for net_key, net_data in nested_data.items():
                        
                        # Skip known non-network metadata fields
                        if net_key in ['clustering_score', 'total_number_edges']:
                            continue
                        
                        if not isinstance(net_data, dict):
                            print(f"Skipping network key '{net_key}' under '{config_key}': not a dictionary.")
                            continue

                        # Create a record for the DataFrame
                        record = {
                            'net_name': net_key,           # e.g., net_53_1113
                            'config_key': config_key,      # e.g., netmap_config_1
                            'filename': filename,
                            # Add the common metadata
                            'clustering_score': clustering_score,
                            'total_number_edges': total_number_edges,
                        }
                        # Add the network-specific data (edge_overlap, net_size, etc.)
                        record.update(net_data)
                        
                        records.append(record)
                    print(records)
                    # Create a DataFrame from the extracted records for this config_key
                    if records:
                        net_dir = op.basename(dirpath)  
                        # The directory one level up is 'CONFIG'
                        config_dir = op.basename(op.dirname(dirpath))
                    


                        overlaps_df = pd.DataFrame(records)
                        # Add the directory name (similar to your original 'net' column)
                        overlaps_df['dataset'] = op.basename(op.dirname(filepath))
                        overlaps_df['config_dir'] = config_dir
                        all_results_dfs.append(overlaps_df)


            except Exception as e:
                print(f"Error reading or processing {filepath}: {e}")
                continue

    if all_results_dfs:
        # Concatenate all DataFrames into one
        all_results = pd.concat(all_results_dfs, ignore_index=True)
    else:
        all_results = pd.DataFrame()

    all_results['n_clusters'] = all_results['config_dir'].apply(recode_number_of_datasets)


    return all_results

def recode_number_of_datasets(config_item):
    if config_item == 'config_easy':
        return 2
    if config_item == 'config_noise':
        return 2
    if config_item == 'config_three':
        return 3
    if config_item == 'config_three_noise':
        return 3
    if config_item == 'config_five':
        return 5
    if config_item == 'config_ten':
        return 10


,net_name,config_key,filename,clustering_score,total_number_edges,edge_overlap,net_size,overlap_percent,reverse_overlap_percent,dataset,config_dir,n_clusters
0,net_84_10865,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,1.000,68644,154,154,1.0,1.0,net_84_10865_net_88_10937_net_90_11013,summaries_best_models_log3,None
1,net_88_10937,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,1.000,68644,184,184,1.0,1.0,net_84_10865_net_88_10937_net_90_11013,summaries_best_models_log3,None
2,net_133_10773,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,1.000,82369,201,201,1.0,1.0,net_133_10773_net_82_10152_net_72_10551,summaries_best_models_log3,None
3,net_82_10152,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,1.000,82369,164,164,1.0,1.0,net_133_10773_net_82_10152_net_72_10551,summaries_best_models_log3,None
4,net_53_11196,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,1.000,42849,78,78,1.0,1.0,net_53_11196_net_70_11431_net_84_9903,summaries_best_models_log3,None
...,...,...,...,...,...,...,...,...,...,...,...,...
95,net_51_10906,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,0.896,203401,90,90,1.0,1.0,net_98_11932_net_51_10906_net_60_10082_net_90_...,summaries_best_models_log3,None
96,net_60_10082,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,0.896,203401,104,104,1.0,1.0,net_98_11932_net_51_10906_net_60_10082_net_90_...,summaries_best_models_log3,None
97,net_75_10306,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,0.896,203401,175,175,1.0,1.0,net_98_11932_net_51_10906_net_60_10082_net_90_...,summaries_best_models_log3,None
98,net_90_11013,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,GradientShap_64_1_0.1_NegativeBinomialAutoenco...,0.896,203401,216,216,1.0,1.0,net_98_11932_net_51_10906_net_60_10082_net_90_...,summaries_best_models_log3,None
